## Example 1: LSP protocol constants

In [ ]:
from pylsp import lsp, hookimpl
from pylsp._utils import find_parents

# Example 1: LSP protocol constants
print("Testing python-lsp-server LSP protocol:")

# LSP protocol enums
print(f"✓ CompletionItemKind: {lsp.CompletionItemKind}")
print(f"  Text: {lsp.CompletionItemKind.Text}")
print(f"  Function: {lsp.CompletionItemKind.Function}")

print(f"✓ DiagnosticSeverity: {lsp.DiagnosticSeverity}")
print(f"  Error: {lsp.DiagnosticSeverity.Error}")
print(f"  Warning: {lsp.DiagnosticSeverity.Warning}")

print(f"✓ SymbolKind: {lsp.SymbolKind}")
print(f"  Function: {lsp.SymbolKind.Function}")
print(f"  Class: {lsp.SymbolKind.Class}")

# Hook implementation
@hookimpl
def test_hook():
    return {"test": "value"}

print(f"✓ Hook decorator: {callable(hookimpl)}")
print(f"✓ Test hook created: {callable(test_hook)}")

# Values fixed by the Language Server Protocol specification
assert (lsp.CompletionItemKind.Text, lsp.CompletionItemKind.Function) == (1, 3)
assert (lsp.DiagnosticSeverity.Error, lsp.DiagnosticSeverity.Warning) == (1, 2)
assert (lsp.SymbolKind.Class, lsp.SymbolKind.Function) == (5, 12)
# hookimpl marks the function as a pylsp plugin hook implementation
assert hasattr(test_hook, "pylsp_impl"), dir(test_hook)
assert test_hook() == {"test": "value"}

## Example 2: Workspace and document management

In [ ]:
from pylsp.workspace import Workspace
import pathlib
import tempfile
import unittest.mock

# Example 2: Workspace and document management
print("Testing python-lsp-server workspace:")

with tempfile.TemporaryDirectory() as temp_dir:
    root = pathlib.Path(temp_dir)
    # The endpoint is the JSON-RPC connection to the editor; a Mock stands in for it here
    workspace = Workspace(root.as_uri(), unittest.mock.Mock())
    assert workspace.root_path == str(root)

    uri = (root / "test.py").as_uri()
    workspace.put_document(uri, "import os\nx = 1\n")
    document = workspace.get_document(uri)
    print(f"✓ Document: {pathlib.Path(document.path).name}, {len(document.lines)} lines")
    assert document.source == "import os\nx = 1\n"
    assert document.lines == ["import os\n", "x = 1\n"]
    assert document.path == str(root / "test.py")
    assert document.word_at_position({"line": 1, "character": 0}) == "x"

    # An incremental change, as an editor sends it, replaces the given range
    change = {"range": {"start": {"line": 1, "character": 4}, "end": {"line": 1, "character": 5}}, "text": "42"}
    workspace.update_document(uri, change, version=2)
    updated = workspace.get_document(uri)
    print(f"✓ After update (version {updated.version}): {updated.source!r}")
    assert updated.source == "import os\nx = 42\n" and updated.version == 2

## Example 3: URI utilities and plugins

In [ ]:
from pylsp import uris
from pylsp.plugins import hover, jedi_completion
import tempfile
import os

# Example 3: URI utilities and plugins
print("Testing python-lsp-server URI and plugins:")

# Test URI conversion
with tempfile.NamedTemporaryFile(suffix='.py', delete=False) as f:
    test_path = f.name
    f.write(b"def test(): pass\n")

try:
    # Path to URI
    file_uri = uris.from_fs_path(test_path)
    print(f"✓ Path to URI: file://...{os.path.basename(test_path)}")
    
    # URI to path
    converted = uris.to_fs_path(file_uri)
    print(f"✓ URI to path: ...{os.path.basename(converted)}")
    print(f"✓ Conversion matches: {test_path == converted}")
    
    # Test plugin
    print(f"✓ Jedi completion plugin: {type(jedi_completion).__name__}")
    print(f"✓ Has pylsp_completions: {hasattr(jedi_completion, 'pylsp_completions')}")
    print(f"✓ Hover plugin has pylsp_hover: {hasattr(hover, 'pylsp_hover')}")
    assert converted == test_path
    assert file_uri.startswith("file://") and file_uri.endswith(os.path.basename(test_path))
    assert callable(jedi_completion.pylsp_completions) and callable(hover.pylsp_hover)
finally:
    os.unlink(test_path)

## Example 4: LSP utilities

In [ ]:
from pylsp._utils import debounce, find_parents
import tempfile
import time
import os

# Example 4: LSP utilities
print("Testing python-lsp-server utilities:")

# Test debounce
calls = []

@debounce(0.1)
def test_func():
    calls.append(time.time())
    return "called"

test_func()
time.sleep(0.5)   # well past the 0.1 s debounce interval
print(f"✓ Debounce executed: {len(calls) > 0}")
assert len(calls) == 1, calls

# Test find_parents
with tempfile.TemporaryDirectory() as temp_dir:
    # Create nested structure
    nested = os.path.join(temp_dir, "src", "package")
    os.makedirs(nested, exist_ok=True)
    
    # Create project files
    open(os.path.join(temp_dir, "setup.py"), 'w').close()
    
    # Find parents - needs path and names
    result = find_parents(temp_dir, nested, ["setup.py"])
    print(f"✓ find_parents result: {[os.path.basename(p) for p in result]}")
    assert result == [os.path.join(temp_dir, "setup.py")], result

## Example 5: LSP protocol types and classes

In [ ]:
from pylsp import lsp
from pylsp.workspace import Workspace, Document

# Example 5: LSP protocol types and classes
print("Testing python-lsp-server types:")

# LSP protocol message types
print(f"✓ MessageType.Error: {lsp.MessageType.Error}")
print(f"✓ MessageType.Warning: {lsp.MessageType.Warning}")
print(f"✓ MessageType.Info: {lsp.MessageType.Info}")

# LSP error codes
print(f"✓ ErrorCodes.ParseError: {lsp.ErrorCodes.ParseError}")
print(f"✓ ErrorCodes.InvalidRequest: {lsp.ErrorCodes.InvalidRequest}")
print(f"✓ ErrorCodes.MethodNotFound: {lsp.ErrorCodes.MethodNotFound}")

# Text document sync kinds
print(f"✓ TextDocumentSyncKind.NONE: {lsp.TextDocumentSyncKind.NONE}")
print(f"✓ TextDocumentSyncKind.FULL: {lsp.TextDocumentSyncKind.FULL}")
print(f"✓ TextDocumentSyncKind.INCREMENTAL: {lsp.TextDocumentSyncKind.INCREMENTAL}")

# Show available classes
print(f"✓ Workspace class: {Workspace}")
print(f"✓ Document class: {Document}")
assert (lsp.MessageType.Error, lsp.MessageType.Warning, lsp.MessageType.Info) == (1, 2, 3)
assert (lsp.ErrorCodes.ParseError, lsp.ErrorCodes.InvalidRequest, lsp.ErrorCodes.MethodNotFound) == (-32700, -32600, -32601)
assert (lsp.TextDocumentSyncKind.NONE, lsp.TextDocumentSyncKind.FULL, lsp.TextDocumentSyncKind.INCREMENTAL) == (0, 1, 2)